# Step 4b — Web Search: Answering with Live Retrieval

[Closed book](04a_closed_book.ipynb) measured what's in the weights. This
notebook adds **one thing** — the provider's built-in search tool — while
keeping the prompt **byte-for-byte identical**. That's deliberate: when two
conditions differ by exactly one thing, the accuracy gap *is* the effect of
that thing.

As in notebook 4a, we start at the raw SDKs — OpenAI, then Gemini — see
exactly what the search tool changes in each response, then hand it all
back to the toolkit — and finish with this method's most audit-relevant
knob: **domain filtering**.

## 1. The baseline: closed book

Same question, same toolkit call as notebook 4a:

In [1]:
from toolkit import prompts
from toolkit.answers import answer_question
from toolkit.utils import load_jsonl

selected = load_jsonl("../../data/questions/selected_questions.jsonl")
question = selected[0]
user_prompt = prompts.build_answer_user_prompt(
    question["question"], question["options"]
)

closed = answer_question(
    question, model="gpt-5.4-mini-2026-03-17", method="closed_book"
)

print(question["question"], "\n")
for letter, option in zip("ABCD", question["options"]):
    mark = "*" if letter == question["correct_letter"] else " "
    print(f"  {mark}{letter}. {option}")
print(f"\nclosed_book: {closed['answer_letter']} "
      f"(confidence {closed['confidence']:.2f}) -> "
      f"{'CORRECT' if closed['is_correct'] else 'WRONG'}")

Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope? 

   A. Breaking News Australia
  *B. Proudly Human
   C. Australian Catholic University
   D. The Vatican Publishing House

closed_book: B (confidence 0.86) -> CORRECT


## 2. Turning on search — the raw OpenAI call

At the SDK level, "web search" is one extra argument: `tools`. OpenAI's
Responses API takes a list of tool specs, and `{"type": "web_search"}` is
the entire spec for the built-in search tool. Everything else — messages,
schema — is unchanged from notebook 4a's raw call:

In [2]:
from openai import OpenAI

from toolkit.answers import Answer
from toolkit.providers import PROVIDER_ENV, load_api_key

client = OpenAI(api_key=load_api_key(PROVIDER_ENV["openai"]))

response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {"role": "developer", "content": prompts.ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ],
    text_format=Answer,
    tools=[{"type": "web_search"}],      # <- the one new argument
)

response.output_parsed

Answer(answer_letter='B', confidence=0.98, reasoning='The review was conducted by Proudly Human, the Australian AI-detection company reported to have certified *Maps of Hope* as human-authored. The other options are either publishers or unrelated outlets.')

Handing a model a tool doesn't force it to use the tool — it can still
answer from its weights. The raw response tells us what actually
happened: its `output` is a list of items, and any item of type
`web_search_call` is a search the model chose to run:

In [3]:
raw = response.model_dump(mode="json", warnings=False)
for item in raw["output"]:
    line = f"  {item['type']}"
    if item["type"] == "web_search_call":
        query = (item.get("action") or {}).get("query")
        line += f"  query={query!r}"
    print(line)

search_used = any(i["type"] == "web_search_call" for i in raw["output"])
print("\nsearch_used =", search_used)

  web_search_call  query='Pope Leo XIV Maps of Hope AI detection review company conducted review speeches writings Maps of Hope'
  message

search_used = True


## 3. The same call on Gemini

Gemini's dialect of the same idea: the built-in tool is called
`google_search`, and it goes inside the `GenerateContentConfig` next to
the schema — again one new line on top of notebook 4a's raw Gemini call.
The search evidence lives in a different place too: not output items,
but `grounding_metadata` on the response candidate, which records the
queries the model actually ran:

In [4]:
from google import genai
from google.genai import types

gclient = genai.Client(api_key=load_api_key(PROVIDER_ENV["gemini"]))

gresponse = gclient.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=prompts.ANSWER_SYSTEM_PROMPT,
        response_mime_type="application/json",
        response_schema=Answer,
        tools=[types.Tool(google_search=types.GoogleSearch())],  # <- the one new line
    ),
)

ganswer = gresponse.parsed
grounding = gresponse.candidates[0].grounding_metadata
queries = grounding.web_search_queries if grounding else None

print(f"Gemini: {ganswer.answer_letter} ({ganswer.confidence:.2f})")
print("search queries:", queries)
print("search_used =", bool(queries))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Gemini: B (1.00)
search queries: None
search_used = False


Notice what just happened above: given the same tool, this Gemini run
*chose not to search* — it was confident enough to answer from its
weights, and the empty `grounding_metadata` proves it. That's evidence,
not assumption, and it's exactly why we record it.

Those two checks — `web_search_call` items for OpenAI,
`grounding_metadata.web_search_queries` for Gemini — are verbatim how the
toolkit's `_detect_search_use()` works. Recording it matters: an
"advantage of search" analysis is meaningless for answers where the
model never searched.

## 4. This is all in the toolkit

`answer_question(method="web_search")` runs the same call — the provider
adapters take a `use_web_search` flag that adds the tool spec — detects
search use, and grades the answer into the standard record:

In [5]:
web = answer_question(
    question, model="gpt-5.4-mini-2026-03-17", method="web_search"
)

print(f"closed_book : {closed['answer_letter']} "
      f"(confidence {closed['confidence']:.2f})")
print(f"web_search  : {web['answer_letter']} "
      f"(confidence {web['confidence']:.2f}, "
      f"searched: {web['search_used']})")
print("WHY:", web["reasoning"])

closed_book : B (confidence 0.86)
web_search  : B (confidence 0.99, searched: True)
WHY: The review was conducted by Proudly Human, the Australian company that certified *Maps of Hope* as human-authored. The article notes it worked with the publisher, ACU, and Vatican cardinals, but the AI detection review itself was done by Proudly Human.


## 5. Parameter deep-dive: choosing where the model can look

"Web search" sounds like one condition, but *which web* matters. OpenAI's
`web_search` tool takes a `filters` object with two lists (up to 100
domains each, written without the `https://` scheme):

```python
tools=[{
    "type": "web_search",
    "filters": {
        "allowed_domains": ["theguardian.com"],   # may ONLY cite these
        # "blocked_domains": ["theguardian.com"], # may cite anything BUT these
    },
}]
```

The toolkit exposes them as `include_domains` and `exclude_domains` on
`answer_question()`. That turns retrieval into a controlled variable, and
our quiz has a perfect stress test built in: **every question was written
from a Guardian article.**

- `include_domains=["theguardian.com"]` points the model straight at the
  source outlet — an open-book exam where we chose the book.
- `exclude_domains=["theguardian.com"]` takes the source away — can the
  model corroborate the fact anywhere *else* on the web?

(One asymmetry to know: this is an OpenAI-only knob. The Gemini
Developer API has no domain filters — `GoogleSearch.exclude_domains`
exists in the SDK but is Enterprise-only — so the toolkit raises a
`ValueError` if you try, rather than silently ignoring the filter.)

In [6]:
included = answer_question(
    question,
    model="gpt-5.4-mini-2026-03-17",
    method="web_search",
    include_domains=["theguardian.com"],
)

print(f"guardian only: {included['answer_letter']} "
      f"(confidence {included['confidence']:.2f}, "
      f"searched: {included['search_used']})")
print("WHY:", included["reasoning"])

guardian only: B (confidence 0.98, searched: True)
WHY: The Guardian reports that Proudly Human undertook the AI detection review of Pope Leo XIV’s *Maps of Hope* collection. That matches option B exactly.


In [7]:
excluded = answer_question(
    question,
    model="gpt-5.4-mini-2026-03-17",
    method="web_search",
    exclude_domains=["theguardian.com"],
)

runs = [
    ("closed_book", closed),
    ("web (unrestricted)", web),
    ("web (guardian only)", included),
    ("web (guardian blocked)", excluded),
]
print(f"correct letter: {question['correct_letter']}\n")
for label, r in runs:
    verdict = "CORRECT" if r["is_correct"] else "WRONG"
    searched = r.get("search_used", "-")
    print(f"{label:<24} {r['answer_letter']} "
          f"(confidence {r['confidence']:.2f}, searched: {searched}) "
          f"-> {verdict}")

correct letter: B

closed_book              B (confidence 0.86, searched: None) -> CORRECT
web (unrestricted)       B (confidence 0.99, searched: True) -> CORRECT
web (guardian only)      B (confidence 0.98, searched: True) -> CORRECT
web (guardian blocked)   B (confidence 0.98, searched: True) -> CORRECT


What to look for:

- **Guardian-only** should be the easiest condition of the whole
  tutorial — the answer is *in* the one domain the model may read. If it
  still misses, that's a retrieval failure, not a knowledge failure.
- **Guardian-blocked** is the interesting audit: it asks whether the fact
  exists *independently* of the outlet that reported it. Wire-service
  pickups, official announcements, and other outlets' coverage often
  keep the model correct — but confidence may drop, and thinly-reported
  stories can flip to wrong.
- **`search_used`** can differ across conditions: with the strongest
  domain restricted, the model sometimes searches more (chasing
  corroboration) or gives up and answers from weights.

This is the general recipe for **provenance-controlled evaluation**:
restrict retrieval to a source list you trust (or distrust) and measure
what changes. Fact-checking sites only, `.gov` only, blocking a suspected
misinformation outlet — same two parameters every time.

## 6. The full experiment, from the command line

The sweep runs unfiltered — the open web is the condition being measured;
the filters above are the notebook's microscope, not part of the
pipeline:

```bash
# 04-1: web search, all six models (600 calls)
for M in gpt-5.4-mini-2026-03-17 gpt-5.5-2026-04-23 gpt-5.6-luna \
         gpt-5.6-terra gemini-3.1-flash-lite gemini-3.5-flash; do
  uv run python scripts/04-1_generate_answers.py \
      --model $M --method web_search --parallel
done
```

## 7. The map

| This notebook | Where it lives |
|---|---|
| §2 the OpenAI tool spec | `toolkit.providers._keys.OPENAI_WEBSEARCH_TOOLS`; the `use_web_search` flag on `openai_provider.run_parsed()` |
| §3 the Gemini tool spec | `toolkit.providers.gemini_provider.GEMINI_WEBSEARCH_TOOLS`; same flag on `gemini_provider.run_parsed()` |
| §2–3 search detection | `toolkit.answers._detect_search_use()` |
| §4 one graded record | `toolkit.answers.answer_question()` |
| §5 domain filters | the `include_domains` / `exclude_domains` kwargs on `answer_question()`; tool built in `toolkit.providers.openai_provider._build_websearch_tools()` |
| §6 at scale | `scripts/04-1_generate_answers.py`; `toolkit.answers.answer_questions()` |

---

### Next up 🗣️

The last method gives the model no tools at all — just company:
[`04c_debate.ipynb`](04c_debate.ipynb) makes three copies of the model
argue it out.